# NutriAdapt — Supervised Fine-Tuning (4-bit QLoRA) & Hugging Face Hub Deployment
## Efficient Fine-Tuned LLM for Adaptive Multimodal Nutrition Recommendation
**Reference paper**: *Personalized Nutrition-Aware Dietary Recommendation With Multimodal LLMs* (IEEE Access, Feb 2026)

### What this notebook does
1. Fine-tunes **Qwen2.5-3B-Instruct** on **either accelerator** (auto-detected):
   - **NVIDIA GPU (e.g. free T4)** → 4-bit NF4 QLoRA (bitsandbytes + PEFT + TRL), fp16
   - **Google TPU** → bf16 LoRA (bitsandbytes is CUDA-only), bf16
2. Builds a multi-task instruction dataset **entirely from the numbers in your recipe table** (no invented provenance or confidence values):
   - Task 1: nutrition extraction
   - Task 2: goal-grounded recommendation explanation (only for recipes that actually meet the goal's thresholds)
   - Task 3: two-meal comparison (answers computed from the data)
   - *Evidence-grounded Q&A is not implemented yet — it needs a real evidence corpus.*
3. Splits train/validation **by recipe** (no leakage between splits).
4. Pushes the LoRA adapter, tokenizer and an honest model card to the Hugging Face Hub.
5. Validates generated JSON with a schema **and** checks every number in the justification against the prompt.

> **Secrets:** add your Hugging Face write token in Colab → 🔑 *Secrets* as `HF_TOKEN` (enable *Notebook access*). Never paste tokens into code cells.


In [1]:
# Mount Google Drive (needed for the recipes parquet and for saving checkpoints / adapter)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Step 1: Install dependencies.
# torch is intentionally NOT upgraded so Colab's CUDA build stays intact.
# If you re-run this after other cells have already imported transformers/trl/peft,
# use Runtime > Restart session once, then continue from Step 2.
%pip install -q -U transformers trl peft bitsandbytes accelerate datasets huggingface_hub jsonschema

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.3.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [3]:
# Step 2: Hardware, environment and paths — uses EXACTLY ONE accelerator: TPU if present, else NVIDIA GPU
import os, gc, re, json, random, inspect
import numpy as np
import pandas as pd
import torch
import transformers, trl, peft

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("=== Hardware & Environment Diagnostic ===")
print(f"torch {torch.__version__} | transformers {transformers.__version__} | trl {trl.__version__} | peft {peft.__version__}")

ACCELERATOR = None          # "tpu" or "cuda" — the ONLY device used for the rest of the notebook
device = None

# 1) Google TPU (torch_xla, or torch_tpu)
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    ACCELERATOR = "tpu"
except Exception:
    try:
        import torch_tpu  # noqa: F401
        device = torch.device("tpu")
        ACCELERATOR = "tpu"
    except Exception:
        pass

# 2) NVIDIA GPU — only if no TPU was found
if ACCELERATOR is None and torch.cuda.is_available():
    device = torch.device("cuda")
    ACCELERATOR = "cuda"

if ACCELERATOR is None:
    raise SystemError(
        "No accelerator found (CPU would run out of memory). In Colab choose: "
        "Runtime > Change runtime type > 'T4 GPU' or 'TPU' > Save, then re-run."
    )

if ACCELERATOR == "tpu":
    compute_dtype = torch.bfloat16            # native TPU precision
    hardware_name = "Google TPU"
    print(f"✓ Using Google TPU: {device}  | compute dtype: {compute_dtype}")
else:
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    compute_dtype = torch.float16             # fp16 on every CUDA GPU (T4 has no fast bf16); one dtype avoids GradScaler clashes
    hardware_name = gpu_name
    print(f"✓ Using NVIDIA GPU: {gpu_name} | VRAM {total_mem:.1f} GB | compute dtype: {compute_dtype}")
print(f"(Only the {ACCELERATOR.upper()} will be used; the other is ignored.)")

# Google Drive (already mounted by the cell above; this is a safe no-op if so)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    candidates = [f"/content/drive/MyDrive/{n}" for n in ("capstone", "CAPSTONE", "Capstone")]
except ImportError:
    print("Not running in Colab – using local folders.")
    candidates = [os.path.abspath(p) for p in ("mydrive/capstone", "mydrive/CAPSTONE", ".")]

BASE_DIR = next((p for p in candidates if os.path.exists(p)), candidates[0])
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
MODEL_OUTPUT_DIR = os.path.join(BASE_DIR, "models", "nutriadapt-qwen-3b")
ADAPTER_DIR = os.path.join(MODEL_OUTPUT_DIR, "final_adapter")
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
print(f"Base directory:   {BASE_DIR}")
print(f"Checkpoint dir:   {MODEL_OUTPUT_DIR}")

=== Hardware & Environment Diagnostic ===
torch 2.11.0+cu130 | transformers 5.20.2 | trl 1.16.0 | peft 0.21.2
✓ Using NVIDIA GPU: Tesla T4 | VRAM 14.6 GB | compute dtype: torch.float16
(Only the CUDA will be used; the other is ignored.)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base directory:   /content/drive/MyDrive/capstone
Checkpoint dir:   /content/drive/MyDrive/capstone/models/nutriadapt-qwen-3b


In [4]:
# Step 3: Hugging Face authentication (token comes from Colab Secrets / env – never hard-code it)
from huggingface_hub import login, HfApi, notebook_login

print("=== Hugging Face Hub Authentication ===")
HF_REPO_NAME = "nutriadapt-qwen-3b-nutrition"
PRIVATE_REPO = False          # set True to keep the repo private until you have validated the model

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("Loaded HF_TOKEN from Colab Secrets.")
    except Exception:
        HF_TOKEN = None

if HF_TOKEN:
    login(token=HF_TOKEN)           # no add_to_git_credential: the token is not written to disk
    print("✓ Logged in to Hugging Face.")
else:
    print("No HF_TOKEN secret found – opening interactive login...")
    notebook_login()

api = HfApi()
who = api.whoami()                  # fails loudly if the token is invalid / read-only
current_user = who["name"]
TARGET_REPO_ID = f"{current_user}/{HF_REPO_NAME}"
print(f"Logged in as: {current_user}")
print(f"Target repo:  https://huggingface.co/{TARGET_REPO_ID}")

api.create_repo(repo_id=TARGET_REPO_ID, repo_type="model", private=PRIVATE_REPO, exist_ok=True)
print("✓ Repository is ready.")

=== Hugging Face Hub Authentication ===
Loaded HF_TOKEN from Colab Secrets.
✓ Logged in to Hugging Face.
Logged in as: narayah90
Target repo:  https://huggingface.co/narayah90/nutriadapt-qwen-3b-nutrition
✓ Repository is ready.


In [5]:
# Step 4: Build the multi-task SFT dataset (every target is derived from real table values)
print("Building supervised fine-tuning instruction dataset...")
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are NutriAdapt, an expert dietary reasoning assistant. Provide factual, evidence-grounded "
    "nutrition advice adhering strictly to the nutrient data given in the prompt. "
    "Never fabricate nutrient values. Return valid JSON when requested."
)

REQUIRED_COLS = ["recipe_name", "ingredients", "calories", "protein_g", "fat_g", "carbs_g", "sodium_mg", "fiber_g"]
NUM_COLS = ["calories", "protein_g", "fat_g", "carbs_g", "sodium_mg", "fiber_g"]
MAX_RECIPES = 2500        # None = use every recipe (training time grows linearly)

# ---------- load ----------
recipe_path = os.path.join(PROCESSED_DIR, "recipes_enriched.parquet")
if os.path.exists(recipe_path):
    df_recipes = pd.read_parquet(recipe_path)
    missing = [c for c in REQUIRED_COLS if c not in df_recipes.columns]
    if missing:
        raise KeyError(f"recipes_enriched.parquet is missing columns {missing}. Found: {list(df_recipes.columns)}")
    print(f"✓ Loaded {len(df_recipes)} recipes from {recipe_path}")
else:
    print("⚠️ recipes_enriched.parquet not found – using the tiny fallback demo table (NOT enough to train a useful model).")
    df_recipes = pd.DataFrame({
        "recipe_name": ["Grilled Salmon Bowl", "Mediterranean Salad", "Chicken Tikka", "Tofu Stir-fry"],
        "ingredients": ["salmon^brown rice^broccoli", "quinoa^tomatoes^cucumber^feta", "chicken^spices^yogurt", "tofu^peppers^soy sauce"],
        "calories": [450.0, 320.0, 520.0, 340.0], "protein_g": [38.0, 12.0, 42.0, 22.0],
        "fat_g": [18.0, 14.0, 20.0, 12.0], "carbs_g": [35.0, 40.0, 18.0, 28.0],
        "sodium_mg": [420.0, 480.0, 680.0, 520.0], "fiber_g": [6.0, 7.0, 3.0, 5.0],
    })

# ---------- clean (NaN would produce invalid JSON targets) ----------
df_recipes = df_recipes[REQUIRED_COLS].copy()
df_recipes[NUM_COLS] = df_recipes[NUM_COLS].apply(pd.to_numeric, errors="coerce")
before = len(df_recipes)
df_recipes = df_recipes.dropna(subset=REQUIRED_COLS)
df_recipes = df_recipes[(df_recipes[NUM_COLS] >= 0).all(axis=1)]
df_recipes = df_recipes.drop_duplicates(subset="recipe_name").reset_index(drop=True)
print(f"✓ Cleaned: {before} -> {len(df_recipes)} recipes (dropped NaN / negative / duplicate names)")

if MAX_RECIPES and len(df_recipes) > MAX_RECIPES:
    df_recipes = df_recipes.sample(n=MAX_RECIPES, random_state=SEED).reset_index(drop=True)
    print(f"✓ Sampled {len(df_recipes)} recipes for efficient Colab fine-tuning.")

# ---------- split BY RECIPE so no recipe appears in both train and validation ----------
perm = np.random.RandomState(SEED).permutation(len(df_recipes))
n_val = max(1, int(round(0.1 * len(df_recipes))))
recipes_val = df_recipes.iloc[perm[:n_val]].reset_index(drop=True)
recipes_train = df_recipes.iloc[perm[n_val:]].reset_index(drop=True)

# ---------- helpers ----------
def fmt(x):
    """380.0 -> '380', 6.25 -> '6.2' (identical formatting in training data, tests and model card)."""
    return f"{float(x):.1f}".rstrip("0").rstrip(".")

def facts(r):
    return f"{fmt(r['calories'])} kcal, {fmt(r['protein_g'])}g protein, {fmt(r['fiber_g'])}g fiber, {fmt(r['sodium_mg'])}mg sodium"

def explain_prompt(name, goal, calories, protein_g, fiber_g, sodium_mg):
    return (f"Explain why '{name}' is recommended for a user with the goal '{goal}'.\n"
            f"Nutritional Facts: {fmt(calories)} kcal, {fmt(protein_g)}g protein, {fmt(fiber_g)}g fiber, {fmt(sodium_mg)}mg sodium.")

def msgs(user, assistant):
    return {"messages": [{"role": "system", "content": SYSTEM_PROMPT},
                         {"role": "user", "content": user},
                         {"role": "assistant", "content": assistant}]}

# Goal rules: a recipe is only "recommended" for a goal if it truly satisfies the thresholds.
GOAL_RULES = {
    "weight_loss":   ("low-calorie and high-fiber",
                      lambda r: r["calories"] <= 500 and r["fiber_g"] >= 4,
                      lambda r: [f"calories {fmt(r['calories'])} <= 500", f"fiber_g {fmt(r['fiber_g'])} >= 4"]),
    "heart_healthy": ("low-sodium (<500mg)",
                      lambda r: r["sodium_mg"] < 500,
                      lambda r: [f"sodium_mg {fmt(r['sodium_mg'])} < 500"]),
    "muscle_gain":   ("high-protein (>30g)",
                      lambda r: r["protein_g"] > 30,
                      lambda r: [f"protein_g {fmt(r['protein_g'])} > 30"]),
}
GOAL_THRESHOLD_NUMBERS = {"500", "4", "30"}   # numbers that may legitimately appear in a justification

def build_samples(df):
    rows = df.to_dict("records")
    out = {"extract": [], "explain": [], "compare": []}

    # Task 1 – nutrition extraction
    for r in rows:
        target = {
            "meal": r["recipe_name"],
            "calories": round(r["calories"], 1),
            "macronutrients": {"protein_g": round(r["protein_g"], 1), "fat_g": round(r["fat_g"], 1), "carbs_g": round(r["carbs_g"], 1)},
            "micronutrients": {"sodium_mg": round(r["sodium_mg"], 1), "fiber_g": round(r["fiber_g"], 1)},
            "source": "NutriAdapt recipes_enriched table",
        }
        out["extract"].append(msgs(
            f"Extract the standardized nutritional breakdown for the following meal:\nMeal: {r['recipe_name']}\nIngredients: {r['ingredients']}",
            json.dumps(target, indent=2)))

    # Task 2 – goal-grounded explanation (only goals the recipe really satisfies)
    for i, r in enumerate(rows):
        matched = [g for g, (_, ok, _) in GOAL_RULES.items() if ok(r)]
        if not matched:
            continue
        goal = matched[i % len(matched)]
        desc, _, crit = GOAL_RULES[goal]
        target = {
            "recommendation": r["recipe_name"],
            "target_goal": goal,
            "nutritional_justification": (
                f"Provides {fmt(r['protein_g'])}g protein and {fmt(r['fiber_g'])}g fiber with {fmt(r['calories'])} kcal "
                f"and {fmt(r['sodium_mg'])}mg sodium, supporting the {desc} goal."),
            "criteria_met": crit(r),
            "factual_compliance": True,
        }
        out["explain"].append(msgs(
            explain_prompt(r["recipe_name"], goal, r["calories"], r["protein_g"], r["fiber_g"], r["sodium_mg"]),
            json.dumps(target, indent=2)))

    # Task 3 – two-meal comparison (answers computed, never guessed)
    def better(a, b, col, higher):
        if a[col] == b[col]:
            return "tie"
        return a["recipe_name"] if (a[col] > b[col]) == higher else b["recipe_name"]
    for a, b in zip(rows[0::2], rows[1::2]):
        target = {
            "meal_a": a["recipe_name"], "meal_b": b["recipe_name"],
            "lower_calorie": better(a, b, "calories", False),
            "higher_protein": better(a, b, "protein_g", True),
            "higher_fiber": better(a, b, "fiber_g", True),
            "lower_sodium": better(a, b, "sodium_mg", False),
        }
        out["compare"].append(msgs(
            f"Compare the following two meals.\nMeal A: {a['recipe_name']} ({facts(a)})\nMeal B: {b['recipe_name']} ({facts(b)})",
            json.dumps(target, indent=2)))
    return out

train_tasks, val_tasks = build_samples(recipes_train), build_samples(recipes_val)
train_samples = [s for v in train_tasks.values() for s in v]
val_samples = [s for v in val_tasks.values() for s in v]
random.Random(SEED).shuffle(train_samples)
random.Random(SEED).shuffle(val_samples)

train_data, val_data = Dataset.from_list(train_samples), Dataset.from_list(val_samples)
print(f"✓ Train: {len(train_data)} examples from {len(recipes_train)} recipes "
      f"(extract {len(train_tasks['extract'])}, explain {len(train_tasks['explain'])}, compare {len(train_tasks['compare'])})")
print(f"✓ Val:   {len(val_data)} examples from {len(recipes_val)} recipes (no recipe overlap with train)")
if len(train_data) < 200:
    print("⚠️ Very small training set – results will not be meaningful.")
print("\nSample training target:")
print(train_data[0]["messages"][2]["content"][:400])

Building supervised fine-tuning instruction dataset...
✓ Loaded 45630 recipes from /content/drive/MyDrive/capstone/processed/recipes_enriched.parquet
✓ Cleaned: 45630 -> 45060 recipes (dropped NaN / negative / duplicate names)
✓ Sampled 2500 recipes for efficient Colab fine-tuning.
✓ Train: 5144 examples from 2250 recipes (extract 2250, explain 1769, compare 1125)
✓ Val:   573 examples from 250 recipes (no recipe overlap with train)

Sample training target:
{
  "recommendation": "White Bean Pate",
  "target_goal": "heart_healthy",
  "nutritional_justification": "Provides 2.8g protein and 2.7g fiber with 89.9 kcal and 258.1mg sodium, supporting the low-sodium (<500mg) goal.",
  "criteria_met": [
    "sodium_mg 258.1 < 500"
  ],
  "factual_compliance": true
}


In [6]:
# Step 5: Load the base model (TPU: bf16 LoRA | GPU: 4-bit QLoRA) and attach LoRA adapters
from packaging import version
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Free memory if this cell is re-run
for _n in ("trainer", "model"):
    if _n in globals():
        del globals()[_n]
gc.collect()
if ACCELERATOR == "cuda":
    torch.cuda.empty_cache()

BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
print(f"Loading base model: {BASE_MODEL_ID} on {ACCELERATOR.upper()} ({compute_dtype})...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# transformers >= 4.56 renamed `torch_dtype` -> `dtype`
dtype_kw = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"

if ACCELERATOR == "tpu":
    # TPU: bitsandbytes is CUDA-only, so use native bfloat16 + LoRA (a 3B model fits easily in TPU HBM)
    TRAIN_METHOD = "bf16 LoRA (no quantization, TPU)"
    print("Configuring for Google TPU: native bfloat16 PEFT LoRA...")
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, **{dtype_kw: torch.bfloat16})
    model = model.to(device)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
else:
    # NVIDIA GPU: 4-bit NF4 QLoRA
    TRAIN_METHOD = "4-bit NF4 QLoRA (GPU)"
    print("Configuring for NVIDIA GPU: 4-bit NF4 QLoRA...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map={"": 0},               # whole model on the single GPU (no accidental CPU offload)
        trust_remote_code=True,
        **{dtype_kw: compute_dtype},
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False        # required with gradient checkpointing

# LoRA adapter configuration (same on TPU and GPU)
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)

# GPU ONLY — ROOT-CAUSE FIX for: NotImplementedError "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'
# fp16 training uses a GradScaler, which cannot unscale bf16 gradients, so trainable (LoRA) weights must be float32.
# (TPU uses bf16 without a GradScaler, so its LoRA weights stay in bf16.)
if ACCELERATOR == "cuda":
    for p in model.parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()

print("\n=== Trainable Parameters ===")
model.print_trainable_parameters()
print("Trainable dtypes:", {str(p.dtype) for p in model.parameters() if p.requires_grad})
if ACCELERATOR == "cuda":
    print(f"GPU memory after load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Loading base model: Qwen/Qwen2.5-3B-Instruct on CUDA (torch.float16)...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


=== Trainable Parameters ===
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Trainable dtypes: {'torch.float32'}
GPU memory after load: 2.62 GB


In [7]:
# Step 6: Train with TRL SFTTrainer (precision/optimizer chosen for the single active accelerator)
from trl import SFTConfig, SFTTrainer
from transformers.trainer_utils import get_last_checkpoint

EPOCHS = 2                     # 2 is usually enough for templated data; raise to 3 if validation loss still falls

# ---- choose max_length from the real token lengths so the JSON answer is never truncated ----
def n_tokens(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return len(tokenizer(text).input_ids)
lens = np.array([n_tokens(train_data[i]) for i in range(min(len(train_data), 1000))])
p99 = int(np.percentile(lens, 99))
MAX_LEN = int(min(1024, max(512, int(np.ceil(p99 / 64) * 64))))
print(f"Token length  mean={lens.mean():.0f}  p99={p99}  max={lens.max()}  ->  max_length={MAX_LEN}")
if lens.max() > MAX_LEN:
    print(f"⚠️ {(lens > MAX_LEN).sum()} sampled examples exceed {MAX_LEN} tokens and will be truncated.")

print(f"=== Training on {ACCELERATOR.upper()} | fp16={ACCELERATOR == 'cuda'} bf16={ACCELERATOR == 'tpu'} ===")

# ---- build config in a way that works on old AND new transformers/TRL ----
cfg_params = inspect.signature(SFTConfig.__init__).parameters
cfg = dict(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="paged_adamw_8bit" if ACCELERATOR == "cuda" else "adamw_torch",   # 8-bit paged Adam is CUDA-only
    fp16=(ACCELERATOR == "cuda"),     # GPU: fp16 + GradScaler
    bf16=(ACCELERATOR == "tpu"),      # TPU: native bf16, no scaler
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=SEED,
)
cfg["eval_strategy" if "eval_strategy" in cfg_params else "evaluation_strategy"] = "epoch"
for key in ("max_length", "max_seq_length"):      # renamed across TRL versions
    if key in cfg_params:
        cfg[key] = MAX_LEN
        break
training_args = SFTConfig(**cfg)

trainer_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = dict(model=model, args=training_args, train_dataset=train_data, eval_dataset=val_data)
trainer_kwargs["processing_class" if "processing_class" in trainer_params else "tokenizer"] = tokenizer
trainer = SFTTrainer(**trainer_kwargs)

if ACCELERATOR == "cuda":
    # The Trainer may touch dtypes while preparing the model, so re-assert float32 LoRA weights right before training.
    for p in trainer.model.parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    bad = {str(p.dtype) for p in trainer.model.parameters() if p.requires_grad} - {"torch.float32"}
    assert not bad, f"Trainable params still in {bad}; fp16 GradScaler will fail."
print(f"✓ Precision check OK on {ACCELERATOR.upper()} | fp16={trainer.args.fp16} bf16={trainer.args.bf16}")

steps_per_epoch = int(np.ceil(len(train_data) / (cfg['per_device_train_batch_size'] * cfg['gradient_accumulation_steps'])))
print(f"≈ {steps_per_epoch} optimizer steps/epoch, {steps_per_epoch * EPOCHS} total")

# Resume automatically after a Colab disconnect (checkpoints live on Drive)
last_ckpt = get_last_checkpoint(MODEL_OUTPUT_DIR) if os.path.isdir(MODEL_OUTPUT_DIR) else None
if last_ckpt:
    print(f"Resuming from checkpoint: {last_ckpt}  (delete the checkpoint-* folders to start fresh)")

print("=== Starting Supervised Fine-Tuning ===")
trainer.train(resume_from_checkpoint=last_ckpt)
final_eval = trainer.evaluate()
print(f"✓ Fine-tuning complete | final eval_loss = {final_eval['eval_loss']:.4f}")

Token length  mean=235  p99=291  max=343  ->  max_length=512
=== Training on CUDA | fp16=True bf16=False ===


Tokenizing train dataset:   0%|          | 0/5144 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5144 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5144 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5144 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/573 [00:00<?, ? examples/s]

✓ Precision check OK on CUDA | fp16=True bf16=False
≈ 643 optimizer steps/epoch, 1286 total
=== Starting Supervised Fine-Tuning ===


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.387488,0.399003,0.387174,0.889341,1205578.000000
2,0.323223,0.413958,0.338876,0.889649,2411156.000000


Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
0.323223,0.399003,2,0.387174,0.889341,2411156.000000


✓ Fine-tuning complete | final eval_loss = 0.3990


In [8]:
# Step 7: Save the adapter locally (Drive) and push adapter + tokenizer + model card in ONE commit
print(f"=== Publishing NutriAdapt to Hugging Face Hub: {TARGET_REPO_ID} ===\n")

os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.save_model(ADAPTER_DIR)        # works for both GPU and TPU (XLA); stores the LoRA adapter
tokenizer.save_pretrained(ADAPTER_DIR)

eval_loss = final_eval["eval_loss"]
model_card = f'''---
language:
- en
license: other
license_name: qwen-research
license_link: https://huggingface.co/{BASE_MODEL_ID}/blob/main/LICENSE
library_name: peft
base_model: {BASE_MODEL_ID}
tags:
- nutrition
- dietary-recommendation
- qlora
- lora
- qwen2.5
- structured-output
---

# NutriAdapt (LoRA adapter for {BASE_MODEL_ID})

A **LoRA adapter** (not a full model) that teaches {BASE_MODEL_ID} to produce **structured JSON** for three nutrition tasks,
using only the nutrient values supplied in the prompt:

1. **Nutrition extraction** – meal name + ingredients -> JSON nutrient breakdown
2. **Goal-grounded explanation** – why a meal fits `weight_loss`, `heart_healthy` or `muscle_gain`
3. **Meal comparison** – which of two meals is lower in calories/sodium or higher in protein/fiber

Inspired by *"Personalized Nutrition-Aware Dietary Recommendation With Multimodal Large Language Models"* (Deng & Tu, IEEE Access, 2026).

## Training details
| | |
|---|---|
| Base model | `{BASE_MODEL_ID}` |
| Method | {TRAIN_METHOD}, LoRA r=16, alpha=32, dropout=0.05 |
| Target modules | q/k/v/o/gate/up/down projections |
| Train examples | {len(train_data)} (from {len(recipes_train)} recipes) |
| Validation examples | {len(val_data)} (from {len(recipes_val)} held-out recipes) |
| Epochs | {EPOCHS} |
| Max sequence length | {MAX_LEN} |
| Final validation loss | {eval_loss:.4f} |
| Hardware | {hardware_name} (Google Colab) |

Training targets are **templated** and computed from a recipe nutrition table. Latency, VRAM and accuracy benchmarks versus other
models have **not** been measured for this adapter.

## Usage
```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("{BASE_MODEL_ID}", dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("{TARGET_REPO_ID}")
model = PeftModel.from_pretrained(base, "{TARGET_REPO_ID}")

messages = [
    {{"role": "system", "content": {json.dumps(SYSTEM_PROMPT)}}},
    {{"role": "user", "content": {json.dumps(explain_prompt("Grilled Salmon Bowl", "muscle_gain", 450, 38, 6, 420))}}},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
```

## Limitations
- Not medical or dietetic advice. Always verify nutrient values against a trusted source.
- The model reasons only over the numbers in the prompt; it does not look up foods.
- Output format and goal thresholds are fixed by the training templates (see `GOAL_RULES` in the training notebook).
- The base model is released under the Qwen Research license; check it before commercial use.
'''

readme_path = os.path.join(ADAPTER_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(model_card)

api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=TARGET_REPO_ID,
    repo_type="model",
    commit_message="Add NutriAdapt LoRA adapter, tokenizer and model card",
)
print("🎉 SUCCESS – adapter is live:")
print(f"👉 https://huggingface.co/{TARGET_REPO_ID}")

=== Publishing NutriAdapt to Hugging Face Hub: narayah90/nutriadapt-qwen-3b-nutrition ===

🎉 SUCCESS – adapter is live:
👉 https://huggingface.co/narayah90/nutriadapt-qwen-3b-nutrition


In [9]:
# Step 8: Live evaluation – JSON schema validation + numeric grounding check
from jsonschema import validate, ValidationError

print(f"=== Live inference with the fine-tuned NutriAdapt adapter ({ACCELERATOR.upper()}) ===\n")

EXPLAIN_SCHEMA = {
    "type": "object",
    "required": ["recommendation", "target_goal", "nutritional_justification", "criteria_met", "factual_compliance"],
    "properties": {
        "recommendation": {"type": "string"},
        "target_goal": {"enum": list(GOAL_RULES)},
        "nutritional_justification": {"type": "string"},
        "criteria_met": {"type": "array", "items": {"type": "string"}},
        "factual_compliance": {"type": "boolean"},
    },
}

def extract_json(text):
    """Strip optional ```json fences and parse."""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    return json.loads(text)

def numbers_in(text):
    return {n.rstrip("0").rstrip(".") if "." in n else n for n in re.findall(r"\d+(?:\.\d+)?", text)}

def generate(messages, max_new_tokens=256):
    model.eval()
    model.config.use_cache = True      # was disabled for training
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device if ACCELERATOR == "tpu" else model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# Use the SAME system prompt and prompt template as training
user_msg = explain_prompt("Mediterranean Quinoa Bowl", "heart_healthy", 380, 14, 8, 280)
response = generate([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}])
print("Generated output:\n" + response)

try:
    parsed = extract_json(response)
    validate(parsed, EXPLAIN_SCHEMA)
    print("\n✓ Valid JSON that matches the schema")
except (json.JSONDecodeError, ValidationError) as err:
    parsed = None
    print(f"\n✗ Schema/JSON check failed: {err}")

if parsed:
    allowed = numbers_in(user_msg) | GOAL_THRESHOLD_NUMBERS
    claimed = numbers_in(parsed["nutritional_justification"] + " " + " ".join(parsed["criteria_met"]))
    invented = claimed - allowed
    print("✓ Every number is grounded in the prompt" if not invented
          else f"✗ Possible hallucinated numbers not present in the prompt: {sorted(invented)}")
    print(f"Factual-compliance flag from model: {parsed.get('factual_compliance')}")

=== Live inference with the fine-tuned NutriAdapt adapter (CUDA) ===

Generated output:
{
  "recommendation": "Mediterranean Quinoa Bowl",
  "target_goal": "heart_healthy",
  "nutritional_justification": "Provides 14g protein and 8g fiber with 380 kcal and 280mg sodium, supporting the low-sodium (<500mg) goal.",
  "criteria_met": [
    "sodium_mg 280 < 500"
  ],
  "factual_compliance": true
}

✓ Valid JSON that matches the schema
✓ Every number is grounded in the prompt
Factual-compliance flag from model: True
